# Conduct Sentiment Analysis on Brandon Sanderson dataset


## Install Packages

In [15]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from IPython.display import display, HTML

## Get Data

In [16]:
OHCO = ['title', 'chapter_id', 'paragraph_id', 'sent_id', 'token_id']
SENTS = OHCO[:4]
PARA = OHCO[:3]
CHAP = OHCO[:2]
BOOK = OHCO[:1]
directory_path = 'C:/Users/mason/Box/Text As Data Final'

In [17]:
CORPUS_file = directory_path + '/Output/BrandonSanderson_CORPUS.csv'
LIB_file = directory_path + '/Output/BrandonSanderson_LIB.csv'
salex_file = directory_path + '/Lexicon/salex_nrc.csv'

## Read in NRC Lexicon

In [18]:
SALEX = pd.read_csv(salex_file).set_index('term_str')
SALEX.columns = [col.replace('nrc_','') for col in SALEX.columns]
SALEX

,anger,anticipation,disgust,fear,joy,negative,positive,sadness,surprise,trust,sentiment
term_str,,,,,,,,,,,
abandon,0,0,0,1,0,1,0,1,0,0,-1
abandoned,1,0,0,1,0,1,0,1,0,0,-1
abandonment,1,0,0,1,0,1,0,1,1,0,-1
abduction,0,0,0,1,0,1,0,1,1,0,-1
aberration,0,0,1,0,0,1,0,0,0,0,-1
...,...,...,...,...,...,...,...,...,...,...,...
young,0,1,0,0,1,0,1,0,1,0,1
youth,1,1,0,1,1,0,1,0,1,0,1
zeal,0,1,0,0,1,0,1,0,1,1,1


## Get LIB and CORPUS

In [19]:
LIB = pd.read_csv(LIB_file).set_index('title').sort_index()
LIB.drop(columns=['Length', 'Total_chapters', 'Total_paragraphs', 'File_path'], inplace=True)
TOKENS = pd.read_csv(CORPUS_file, index_col=0).set_index(OHCO)
TOKENS.drop(columns=['token_str'], inplace=True)
TOKENS = TOKENS.join(LIB)
TOKENS.head()

term_str  pos  \
title             chapter_id paragraph_id sent_id token_id                 
A Memory of Light 0          0            0       0             the   DT   
                                                  1           wheel  NNP   
                                                  2              of   IN   
                                                  3            time  NNP   
                                                  4           turns  NNS   

                                                           pos_group  \
title             chapter_id paragraph_id sent_id token_id             
A Memory of Light 0          0            0       0               DT   
                                                  1               NN   
                                                  2               IN   
                                                  3               NN   
                                                  4               NN   

                                                                                       Author  \
title             chapter_id paragraph_id sent_id token_id                                      
A Memory of Light 0          0            0       0         Robert Jordan & Brandon Sanderson   
                                                  1         Robert Jordan & Brandon Sanderson   
                                                  2         Robert Jordan & Brandon Sanderson   
                                                  3         Robert Jordan & Brandon Sanderson   
                                                  4         Robert Jordan & Brandon Sanderson   

                                                                  Date  
title             chapter_id paragraph_id sent_id token_id              
A Memory of Light 0          0            0       0         2013-01-08  
                                                  1         2013-01-08  
                                                  2         2013-01-08  
                                                  3         2013-01-08  
                                                  4         2013-01-08

In [20]:
print("\n".join(list(LIB.index)))

A Memory of Light
Arcanum Unbounded
Elantris
Isles of the Emberdark
Oathbringer
Rhythm of War
Shadows of Self
Steelheart
The Aether of Night
The Alloy of Law
The Bands of Mourning
The Final Empire
The Hero of Ages
The Lost Metal
The Sunlit Man
The Way of Kings
The Well of Ascension
The Wheel of Time
Towers of Midnight
Tress of the Emerald Sea
Warbreaker
Wind and Truth
Words of Radiance
Yumi and the Nightmare Painter


## Sentiment on VOCAB

Sentiment values associated with a subset of the VOCAB from a curated sentiment lexicon.

In [21]:
VOCAB = TOKENS['term_str'].value_counts().to_frame('n')
VOCAB.index.name = 'term_str'
VOCAB_SENT = VOCAB.join(SALEX, how='inner')
VOCAB_SENT

,n,anger,anticipation,disgust,fear,joy,negative,positive,sadness,surprise,trust,sentiment
term_str,,,,,,,,,,,,
don,3334,0,0,0,0,0,0,1,0,0,1,1
good,2309,0,1,0,0,1,0,1,0,1,1,1
found,1607,0,0,0,0,1,0,1,0,0,1,1
lord,1585,0,0,1,0,0,1,1,0,0,1,0
finally,955,0,1,1,0,1,0,1,0,1,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...
nether,1,1,0,0,1,0,1,0,1,0,0,-1
abandonment,1,1,0,0,1,0,1,0,1,1,0,-1
irrefutable,1,0,0,0,0,0,0,1,0,0,1,1


In [22]:
VOCAB_SENT.to_csv(directory_path + '/Output/BrandonSanderson_VOCAB_SENT.csv')

## VOCAB Sentiment on Bag of Words at the Paragraph level

In [23]:
BOW = TOKENS.groupby(PARA+['term_str']).term_str.count().to_frame('n')
emo_cols = "anger anticipation disgust fear joy sadness surprise trust sentiment".split()
BOW_SENT = BOW.join(VOCAB_SENT[emo_cols], on='term_str', how='inner')

for col in emo_cols:
    BOW_SENT[col] = BOW_SENT[col] * BOW_SENT['n']
BOW_SENT

n  anger  \
title                          chapter_id paragraph_id term_str              
A Memory of Light              0          0            birth      1      0   
                                                       forgotten  1      0   
                                          1            bark       1      1   
                                                       disease    1      1   
                                                       found      1      0   
...                                                              ..    ...   
Yumi and the Nightmare Painter 0          42           kiss       1      0   
                                                       real       1      0   
                                                       serve      1      0   
                                                       share      1      0   
                                          44           kiss       1      0   

                                                                  anticipation  \
title                          chapter_id paragraph_id term_str                  
A Memory of Light              0          0            birth                 1   
                                                       forgotten             0   
                                          1            bark                  0   
                                                       disease               0   
                                                       found                 0   
...                                                                        ...   
Yumi and the Nightmare Painter 0          42           kiss                  1   
                                                       real                  0   
                                                       serve                 0   
                                                       share                 1   
                                          44           kiss                  1   

                                                                  disgust  \
title                          chapter_id paragraph_id term_str             
A Memory of Light              0          0            birth            0   
                                                       forgotten        0   
                                          1            bark             0   
                                                       disease          1   
                                                       found            0   
...                                                                   ...   
Yumi and the Nightmare Painter 0          42           kiss             0   
                                                       real             0   
                                                       serve            0   
                                                       share            0   
                                          44           kiss             0   

                                                                  fear  joy  \
title                          chapter_id paragraph_id term_str               
A Memory of Light              0          0            birth         1    1   
                                                       forgotten     1    0   
                                          1            bark          0    0   
                                                       disease       1    0   
                                                       found         0    1   
...                                                                ...  ...   
Yumi and the Nightmare Painter 0          42           kiss          0    1   
                                                       real          0    0   
                                                       serve         0    0   
                                                       share         0    1   
                                  

In [24]:
BOW_SENT.to_csv(directory_path + '/Output/BrandonSanderson_BOW_SENT.csv')

## Document Sentiment - Each Chapter

In [25]:
DOC_SENT = BOW_SENT.groupby(CHAP)[emo_cols].sum()
DOC_SENT

anger  anticipation  disgust  fear  \
title                          chapter_id                                       
A Memory of Light              0              92            88       54   128   
                               1             106            79       62   131   
                               2              78            66       50    97   
                               3              88            62       63    98   
                               4             143           138       79   160   
...                                          ...           ...      ...   ...   
Words of Radiance              4              38            42       28    43   
                               5               5             8        3    11   
                               6              42            53       28    65   
                               7              95            62       40    82   
Yumi and the Nightmare Painter 0              10            25        8    12   

                                           joy  sadness  surprise  trust  \
title                          chapter_id                                  
A Memory of Light              0           102       92        51    152   
                               1            90      106        59    138   
                               2            78      105        52    118   
                               3            54       92        34     94   
                               4           131      127        64    219   
...                                        ...      ...       ...    ...   
Words of Radiance              4            46       50        28     57   
                               5            12        3         0      7   
                               6            48       70        31     60   
                               7            53       67        46     73   
Yumi and the Nightmare Painter 0            26       15        18     27   

                                           sentiment  
title                          chapter_id             
A Memory of Light              0                 -32  
                               1                 -38  
                               2                 -36  
                               3                 -69  
                               4                  -6  
...                                              ...  
Words of Radiance              4                  -7  
                               5                  -1  
                               6                 -34  
                               7                 -42  
Yumi and the Nightmare Painter 0                  20  

[646 rows x 9 columns]

In [26]:
DOC_SENT.to_csv(directory_path + '/Output/BrandonSanderson_DOC_SENT.csv')

# Plotting sentiment over narrative time

In [27]:
plot_df = DOC_SENT.reset_index()
plot_df['max_chapter'] = plot_df.groupby('title')['chapter_id'].transform(max)
plot_df['book_progress'] = plot_df['chapter_id'] / plot_df['max_chapter'].replace(0, 1)

books_to_plot = ['The Wheel of Time', 'The Way of Kings', 'The Final Empire']

plot_df['smoothed_sentiment'] = plot_df.groupby('title')['sentiment'].transform(lambda x: x.rolling(window=3, min_periods=1).mean())

subset_df  = plot_df[plot_df['title'].isin(books_to_plot)].copy()

fig = px.line(
    subset_df,
    x='book_progress',
    y='smoothed_sentiment',
    color='title',
    title='Sentiment Over Normalized Narrative Time',
    labels={
        'book_progress': 'Normalized Narrative Time',
        'smoothed_sentiment': 'Sentiment (Smoothed)',
        'title': 'Title'
    }
)
fig.add_hline(y=0, line_dash='dash', line_color='gray', opacity=0.5)
fig.update_layout(
    xaxis_title='Normalized Narrative Time',
    yaxis_title='Sentiment (Smoothed)',
    legend_title='Title',
    xaxis_tickformat='.2%'
)
fig.show()
fig.write_image(directory_path + '/Output/SentimentOverTime.png')